# Lab 32 (solution): Self-RAG from scratch

Reference implementation. Builds on Lab 06's retrieval stack and adds Self-RAG's **reflection tokens**: an on-demand retrieve decision, per-passage relevance (ISREL), answer support (ISSUP), and usefulness (ISUSE).

Pattern source: Asai et al. 2023, *Self-RAG* ([arXiv:2310.11511](https://arxiv.org/abs/2310.11511), ICLR 2024).

**Key honesty note:** the paper *fine-tunes* a model to emit reflection tokens during generation. We cannot fine-tune in a lab, so we approximate each token with a constrained classification call. This is the documented prompt-based approximation: it captures the control flow and the decisions, but loses the efficiency of inline token emission. Treat it as a faithful model of the *logic*, not the *implementation*.

## Step 0: Setup

In [ ]:
import hashlib
import json
import os
import pathlib
import re
from typing import Any
from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")
PROVIDER = "openai"
MODEL = {"openai": "gpt-4o-mini", "anthropic": "claude-haiku-4-5-20251001"}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")

In [ ]:
def chat(messages: list[dict], temperature: float = 0.0) -> str:
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=MODEL, messages=messages, temperature=temperature)
        return resp.choices[0].message.content or ""
    else:
        from anthropic import Anthropic
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        resp = Anthropic().messages.create(
            model=MODEL, system=system, messages=non_system,
            max_tokens=1024, temperature=temperature)
        return "".join(b.text for b in resp.content if hasattr(b, "text"))


def chat_token(messages: list[dict], allowed: list[str]) -> str:
    """Self-RAG reflection tokens are constrained categorical outputs. We
    approximate by asking for one of `allowed` and snapping to the closest.

    Match on WORD BOUNDARIES, not substrings: "relevant" is a substring of
    "irrelevant", so naive `in` matching would mis-grade every irrelevant
    passage as relevant. \b handles the underscore tokens too (no_retrieve)."""
    raw = chat(messages).strip().lower()
    for tok in allowed:
        if re.search(rf"\b{re.escape(tok.lower())}\b", raw):
            return tok
    return allowed[-1]  # conservative default

## Step 1: Reuse Lab 06's retrieval stack

Same corpus, chunker, and index as Lab 06 / Lab 31.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

CORPUS_DIR = pathlib.Path("../06-agentic-rag-from-scratch/corpus")
def approx_tokens(t): return int(len(t.split()) / 0.75)
def split_paras(t): return [p.strip() for p in re.split(r"\n\s*\n", t) if p.strip()]
def split_sents(t): return [p.strip() for p in re.split(r"(?<=[.!?])\s+", t) if p.strip()]

def chunk_text(text, target=160):
    out, cur, ct = [], [], 0
    for para in split_paras(text):
        pt = approx_tokens(para)
        if ct + pt > target and cur:
            out.append("\n\n".join(cur))
            cur, ct = [], 0
        if pt > target:
            for s in split_sents(para):
                st = approx_tokens(s)
                if ct + st > target and cur:
                    out.append(" ".join(cur))
                    cur, ct = [], 0
                cur.append(s)
                ct += st
        else:
            cur.append(para)
            ct += pt
    if cur:
        out.append("\n\n".join(cur))
    return out

all_chunks = []
for path in sorted(CORPUS_DIR.glob("*.md")):
    if path.name == "README.md":
        continue
    body = path.read_text()
    title = body.splitlines()[0].lstrip("# ").strip()
    for i, ch in enumerate(chunk_text(body)):
        all_chunks.append({"doc_id": path.stem, "chunk_id": f"{path.stem}#{i}",
                           "title": title, "text": ch})
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
embeddings = embedder.encode([c["text"] for c in all_chunks],
                             normalize_embeddings=True, convert_to_numpy=True,
                             show_progress_bar=False)
def search_corpus(query, top_k=5):
    q = embedder.encode([query], normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=False)[0]
    s = embeddings @ q
    idx = np.argsort(s)[::-1][:top_k]
    return [{**all_chunks[i], "score": float(s[i])} for i in idx]
print(f"Index ready: {len(all_chunks)} chunks")

## Step 2: The Retrieve decision (on-demand retrieval)

Static RAG always retrieves. Self-RAG first decides whether retrieval is needed at all — skipping it for queries answerable from parametric knowledge.

In [ ]:
def decide_retrieve(query: str) -> str:
    """Self-RAG\'s Retrieve token: should we retrieve at all? Some queries are
    answerable from parametric knowledge; retrieving for them only adds latency
    and distractors. Returns "retrieve" or "no_retrieve".

    The paper trains this as a special token; we approximate with a constrained
    classification call."""
    return chat_token([
        {"role": "system", "content":
         "Decide if answering the query needs retrieval from a documentation "
         "corpus about AI agents, or can be answered from general knowledge. "
         "Answer with exactly one word: retrieve OR no_retrieve."},
        {"role": "user", "content": query},
    ], allowed=["retrieve", "no_retrieve"])

## Step 3: The reflection tokens (ISREL, ISSUP, ISUSE)

Three graders: is each passage relevant, is a candidate answer supported by its passage, and how useful is the answer. Each is a constrained categorical call here.

In [ ]:
def grade_relevance(query: str, chunk: dict) -> str:
    """ISREL token: is this passage relevant to the query? -> relevant|irrelevant"""
    return chat_token([
        {"role": "system", "content": "Is the passage relevant to the query? "
         "Answer one word: relevant OR irrelevant."},
        {"role": "user", "content": f"Query: {query}\n\nPassage: {chunk['text']}"},
    ], allowed=["relevant", "irrelevant"])


def grade_support(answer: str, chunk: dict) -> str:
    """ISSUP token: is the answer supported by this passage?
    -> fully|partially|no_support"""
    return chat_token([
        {"role": "system", "content": "Is the answer supported by the passage? "
         "Answer one of: fully, partially, no_support."},
        {"role": "user", "content": f"Passage: {chunk['text']}\n\nAnswer: {answer}"},
    ], allowed=["fully", "partially", "no_support"])


def grade_usefulness(query: str, answer: str) -> int:
    """ISUSE token: how useful is the answer to the query? -> 1..5"""
    tok = chat_token([
        {"role": "system", "content": "Rate how useful the answer is for the "
         "query on a 1-5 scale. Answer with a single digit 1, 2, 3, 4, or 5."},
        {"role": "user", "content": f"Query: {query}\n\nAnswer: {answer}"},
    ], allowed=["5", "4", "3", "2", "1"])
    return int(tok)

## Step 4: Per-passage generation

Self-RAG generates one candidate per relevant passage, then chooses among them by reflection score.

In [ ]:
def generate_from_passage(query: str, chunk: dict) -> str:
    """Generate a candidate answer grounded in ONE passage. Self-RAG generates a
    candidate per relevant passage, then picks the best by its reflection scores."""
    return chat([
        {"role": "system", "content": "Answer the query using only the passage. "
         "Cite the passage id in brackets. Be concise."},
        {"role": "user", "content":
         f"Passage [{chunk['chunk_id']}]: {chunk['text']}\n\nQuery: {query}"},
    ])


def generate_parametric(query: str) -> str:
    """No-retrieval path: answer from the model\'s own knowledge."""
    return chat([
        {"role": "system", "content": "Answer concisely from your own knowledge."},
        {"role": "user", "content": query},
    ])

## Step 5: The Self-RAG loop

Decide → retrieve → grade relevance → generate per relevant passage → score by support + usefulness → pick best.

In [ ]:
def self_rag(query: str, top_k: int = 4, verbose: bool = True) -> dict:
    """Self-RAG loop:
      1. decide_retrieve -> maybe skip retrieval entirely
      2. retrieve, grade each passage (ISREL), keep relevant
      3. generate one candidate per relevant passage
      4. score each candidate by support (ISSUP) + usefulness (ISUSE)
      5. return the best-scoring candidate
    Returns the answer plus the reflection trace."""
    trace = []
    decision = decide_retrieve(query)
    trace.append(("decide_retrieve", decision))
    if verbose:
        print(f"  Retrieve? -> {decision}")

    if decision == "no_retrieve":
        ans = generate_parametric(query)
        use = grade_usefulness(query, ans)
        trace.append(("ISUSE", use))
        return {"query": query, "answer": ans, "retrieved": False,
                "usefulness": use, "trace": trace}

    chunks = search_corpus(query, top_k=top_k)
    candidates = []
    for c in chunks:
        rel = grade_relevance(query, c)
        trace.append(("ISREL", c["chunk_id"], rel))
        if rel != "relevant":
            continue
        ans = generate_from_passage(query, c)
        sup = grade_support(ans, c)
        use = grade_usefulness(query, ans)
        trace.append(("candidate", c["chunk_id"], f"ISSUP={sup}", f"ISUSE={use}"))
        # Self-RAG scoring: weight support and usefulness.
        sup_w = {"fully": 1.0, "partially": 0.5, "no_support": 0.0}[sup]
        score = sup_w + use / 5.0
        candidates.append({"answer": ans, "chunk_id": c["chunk_id"],
                           "support": sup, "usefulness": use, "score": score})

    if not candidates:
        # All retrieved passages graded irrelevant -> fall back to parametric.
        ans = generate_parametric(query)
        trace.append(("fallback", "no_relevant_passages"))
        return {"query": query, "answer": ans, "retrieved": True,
                "candidates": 0, "trace": trace}

    best = max(candidates, key=lambda c: c["score"])
    if verbose:
        print(f"  {len(candidates)} candidate(s); best from {best['chunk_id']} "
              f"(ISSUP={best['support']}, ISUSE={best['usefulness']})")
    return {"query": query, "answer": best["answer"], "retrieved": True,
            "best": best, "n_candidates": len(candidates), "trace": trace}

## Step 6: See the two signature behaviors

Self-RAG skips retrieval on a parametric query, and grades+selects on an in-corpus query.

In [ ]:
# Self-RAG\'s signature behaviors:
#  (1) it SKIPS retrieval for queries answerable from parametric knowledge
#  (2) it GRADES each passage and only generates from relevant ones
#  (3) it picks the best candidate by support + usefulness

print("=== Query A: parametric (expect no_retrieve) ===")
a = self_rag("What does the acronym LLM stand for?")
print(f"  retrieved={a['retrieved']}  answer={a['answer'][:120]}\n")

print("=== Query B: in-corpus (expect retrieve + graded candidates) ===")
b = self_rag("How does the ReAct loop interleave reasoning and acting?")
print(f"  retrieved={b['retrieved']}  answer={b['answer'][:160]}")
print(f"  reflection trace had {len(b['trace'])} steps")

## What you built

A from-scratch Self-RAG loop: on-demand retrieval, per-passage relevance grading, and support/usefulness-weighted candidate selection. The cost is several extra grading calls per query — Self-RAG trades calls for fewer hallucinations and fewer needless retrievals.

**Where this implementation simplifies:** the reflection tokens are approximated with classification calls rather than fine-tuned inline tokens (so this is more expensive than the paper's method); candidate generation is one-per-passage (the paper supports richer segment-level decoding). Both are reasonable next extensions.

See [`concepts/rag/sota-rag-patterns.md`](../../../concepts/rag/sota-rag-patterns.md) (Pattern 1).